In [30]:
!pip install \
    llama-index==0.12.0 \
    llama-index-llms-huggingface \
            llama-index-embeddings-huggingface \
            chromadb \
            pinecone-client \
            flask \
            lxml \
            gitpython


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
import re
import json
from git import Repo
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, Document
from pathlib import Path
from lxml import etree
from flask import Flask, request, jsonify, render_template_string, Response

In [32]:
QWEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

Settings.llm = HuggingFaceLLM(
    model_name=QWEN_MODEL,
    tokenizer_name=QWEN_MODEL,
    device_map="auto",  # or "cpu"
    max_new_tokens=512,
    generate_kwargs={"temperature": 0.1, "do_sample": False},
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL
)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\pwdev\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pwdev\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [33]:
BASE_DIR = "./regs"
os.makedirs(BASE_DIR, exist_ok=True)

FAR_REPO_URL = "https://github.com/GSA/GSA-Acquisition-FAR.git"
DFARS_REPO_URL = "https://github.com/GSA/GSA-Acquisition-DFARS.git"

FAR_PATH = os.path.join(BASE_DIR, "far")
DFARS_PATH = os.path.join(BASE_DIR, "dfars")

def clone_if_needed(url, path):
    if not os.path.exists(path):
        print(f"Cloning {url} -> {path}")
        Repo.clone_from(url, path)
    else:
        print(f"Using existing repo at {path}")

clone_if_needed(FAR_REPO_URL, FAR_PATH)
clone_if_needed(DFARS_REPO_URL, DFARS_PATH)

Cloning https://github.com/GSA/GSA-Acquisition-FAR.git -> ./regs\far
Cloning https://github.com/GSA/GSA-Acquisition-DFARS.git -> ./regs\dfars


In [42]:
def parse_ditamap(ditamap_path: Path, regulation: str):
    """
    Parse far.ditamap (or dfars.ditamap) and return:
    {href -> {'fill_types': [...], 'navtitle': ...}}
    NOTE: no namespaces; GitHub FAR DITA often has plain <topicref>.
    """
    parser = etree.XMLParser(recover=True)
    tree = etree.parse(str(ditamap_path), parser=parser)
    root = tree.getroot()

    topic_info = {}

    # no namespaces: just //topicref
    for topicref in root.xpath(".//topicref"):
        href = topicref.get("href")
        if not href:
            continue

        navtitle = topicref.get("navtitle")
        xtrc = topicref.get("xtrc", "")  # e.g. "GFI", "GFI Editable", "VFI"
        fill_types = xtrc.split() if xtrc else []

        topic_info[href] = {
            "regulation": regulation,
            "navtitle": navtitle,
            "fill_types": fill_types,
        }

    return topic_info


def extract_fillins_and_text(dita_path: Path):
    """
    Extract plain text + fill-in metadata from a .dita file.
    - <cite> with xtrf="GFI"/"VFI"/"Editable"
    - outputclass: SingleLine / MultiLine / Checkbox
    - rev attributes (FAC tracking)
    """
    parser = etree.XMLParser(recover=True)
    tree = etree.parse(str(dita_path), parser=parser)
    root = tree.getroot()

    # full text
    text = " ".join(root.itertext())

    # fill-ins
    fillins = []
    for cite in root.xpath(".//cite"):
        xtrf = cite.get("xtrf")
        outputclass = cite.get("outputclass")
        raw = "".join(cite.itertext()).strip()
        if xtrf:
            fillins.append({
                "type": xtrf,           # GFI / VFI / Editable
                "format": outputclass,  # SingleLine / MultiLine / Checkbox
                "placeholder": raw,
            })

    # rev markers
    rev_markers = []
    for el in root.xpath(".//*[@rev]"):
        rev = el.get("rev")
        snippet = "".join(el.itertext()).strip()
        if rev:
            rev_markers.append({
                "rev": rev,   # e.g. "FAC 2025-03 January 17, 2025"
                "text": snippet,
            })

    return text, fillins, rev_markers


def infer_part_subpart_section_from_filename(href: str):
    """
    Heuristic based on FAR naming:
    - 1.101.dita        -> section 1.101
    - 1.102-1.dita      -> section 1.102-1
    - 52.203-14.dita    -> clause 52.203-14
    - part-1.dita       -> part 1
    - subpart-1.1.dita  -> subpart 1.1
    """
    name = Path(href).stem  # e.g., "1.101", "1.102-1", "52.203-14", "part-1"

    part = None
    subpart = None
    section = None
    clause = None

    if name.startswith("part-"):
        part = name.split("part-")[1]
    elif name.startswith("subpart-"):
        subpart = name.split("subpart-")[1]
        part = subpart.split(".")[0]
    else:
        # section or clause
        if re.match(r"^\d+\.\d+(-\d+)?$", name):
            # 1.101 or 1.102-1
            section = name
            part = name.split(".")[0]
        elif re.match(r"^\d+\.\d+-\d+.*", name):
            # 52.203-14 etc.
            clause = name
            part = name.split(".")[0]

    return part, subpart, section, clause


In [43]:
def build_documents_from_repo(repo_path: str, regulation: str):
    """
    - Find main far.ditamap / dfars.ditamap somewhere under the repo
    - Parse topicrefs (xtrc, navtitle, href)
    - For each referenced .dita, build a Document with rich metadata
    """
    repo_path = Path(repo_path)

    # find all .ditamap files
    ditamaps = list(repo_path.rglob("*.ditamap"))
    if not ditamaps:
        print(f"No .ditamap found in {repo_path}")
        return []

    # prefer far.ditamap / dfars.ditamap by name
    main_ditamap = None
    target_name = "far.ditamap" if regulation == "FAR" else "dfars.ditamap"
    for dm in ditamaps:
        if dm.name.lower() == target_name:
            main_ditamap = dm
            break
    if main_ditamap is None:
        # fallback: just take the first one
        main_ditamap = ditamaps[0]

    print(f"[{regulation}] Using DITA map: {main_ditamap}")

    topic_info = parse_ditamap(main_ditamap, regulation)

    docs = []
    for href, info in topic_info.items():
        # href is relative to the ditamap location
        dita_file = (main_ditamap.parent / href).resolve()

        # fallback: sometimes href paths are odd; try repo root as backup
        if not dita_file.exists():
            alt = (repo_path / href).resolve()
            if alt.exists():
                dita_file = alt
            else:
                # skip if we truly can't find it
                continue

        text, fillins, rev_markers = extract_fillins_and_text(dita_file)
        if not text.strip():
            continue

        part, subpart, section, clause = infer_part_subpart_section_from_filename(href)

        metadata = {
            "regulation": regulation,
            "href": href,
            "navtitle": info.get("navtitle"),
            "fill_types": info.get("fill_types", []),  # from xtrc in map
            "fillins": fillins,                        # from <cite> xtrf
            "rev_markers": rev_markers,                # FAC tracking
            "part": part,
            "subpart": subpart,
            "section": section,
            "clause": clause,
            "source_path": str(dita_file),
        }

        docs.append(Document(text=text, metadata=metadata))

    return docs


In [54]:
far_docs = build_documents_from_repo(FAR_PATH, "FAR")
dfars_docs = build_documents_from_repo(DFARS_PATH, "DFARS")

documents = far_docs + dfars_docs
len(documents), documents[0].metadata


ERROR! Session/line number was not unique in database. History logging moved to new session 145
[FAR] Using DITA map: regs\far\dita\FAR.ditamap
[DFARS] Using DITA map: regs\dfars\dita\DFARS-FixDitamapAttributes.ditamap


(7181,
 {'regulation': 'FAR',
  'href': 'Volume_I.dita',
  'navtitle': 'Volume I-Parts 1 to 51',
  'fill_types': ['Not'],
  'fillins': [],
  'rev_markers': [],
  'part': None,
  'subpart': None,
  'section': None,
  'clause': None,
  'source_path': 'C:\\Users\\pwdev\\anaconda_projects\\RAG_Acq\\regs\\far\\dita\\Volume_I.dita'})

In [55]:
def normalize_metadata(md: dict):
    """
    Convert all metadata values into Chroma/Pinecone-safe strings.
    - None -> "None"
    - lists/dicts -> JSON string
    - everything else -> str()
    """
    safe = {}
    for k, v in md.items():
        if v is None:
            safe[k] = "None"
        elif isinstance(v, (list, dict)):
            safe[k] = json.dumps(v)
        else:
            safe[k] = str(v)
    return safe


In [56]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=8192, chunk_overlap=100)
nodes = parser.get_nodes_from_documents(documents)
for node in nodes:
    node.metadata = normalize_metadata(node.metadata)

# Metadata is automatically propagated from Document to each Node
len(nodes), nodes[0].metadata


(7209,
 {'regulation': 'FAR',
  'href': 'Volume_I.dita',
  'navtitle': 'Volume I-Parts 1 to 51',
  'fill_types': '["Not"]',
  'fillins': '[]',
  'rev_markers': '[]',
  'part': 'None',
  'subpart': 'None',
  'section': 'None',
  'clause': 'None',
  'source_path': 'C:\\Users\\pwdev\\anaconda_projects\\RAG_Acq\\regs\\far\\dita\\Volume_I.dita'})

In [57]:
# 1. Configuration Constants
QWEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # CPU-friendly-ish
EMBED_MODEL = "BAAI/bge-small-en-v1.5"     # 384-dimension local embeddings

# 2. Initialize Local Models
# Configure LLM for causal language generation
llm = HuggingFaceLLM(
    model_name=QWEN_MODEL,
    tokenizer_name=QWEN_MODEL,
    device_map="auto",  # Set to "cpu" if you don't have a GPU
    max_new_tokens=512,
    generate_kwargs={"temperature": 0.1, "do_sample": False},
)

# Configure Local Embedding Model
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)

# 3. Apply Global Settings
# Applying to 'Settings' ensures these models are used for both indexing and querying
Settings.llm = llm
Settings.embed_model = embed_model
Settings.text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=100) #

# 4. Setup Local Vector Store (ChromaDB)
chroma_client = chromadb.PersistentClient(path="./chroma_fardfars")
chroma_collection = chroma_client.get_or_create_collection("far_dfars_chroma")

chroma_vs = ChromaVectorStore(chroma_collection=chroma_collection)
chroma_storage = StorageContext.from_defaults(vector_store=chroma_vs)

# 5. Build Index and Query Engine
# This will now use BGE embeddings and Qwen LLM automatically
chroma_index = VectorStoreIndex(
    nodes,
    storage_context=chroma_storage,
    show_progress=True,
)

chroma_qe = chroma_index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact",
)

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1065 [00:00<?, ?it/s]

In [58]:
# 1. Initialize Pinecone Client
PINECONE_API_KEY = os.getenv("Pinecone")
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "far-dfars-rag"

# 2. Create Index if it doesn't exist
# dimension=384 is critical to match BAAI/bge-small-en-v1.5
if index_name not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# 3. Connect to the Index and Setup LlamaIndex Vector Store
pinecone_index = pc.Index(index_name)
# 'text_key' is required to tell Pinecone where to store the actual text
pinecone_vs = PineconeVectorStore(pinecone_index=pinecone_index)
pinecone_storage = StorageContext.from_defaults(vector_store=pinecone_vs)

# 4. Build Index (Uses local BGE embeddings from Settings)
pinecone_index_llama = VectorStoreIndex(
    nodes,
    storage_context=pinecone_storage,
    show_progress=True,
)

# 5. Query Engine (Uses local Qwen-1.5B from Settings)
pinecone_qe = pinecone_index_llama.as_query_engine(
    similarity_top_k=5,
    response_mode="compact",
)

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Upserted vectors:   0%|          | 0/2048 [00:00<?, ?it/s]

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 04 May 2026 18:35:10 GMT', 'Content-Type': 'application/json', 'Content-Length': '115', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '386', 'x-envoy-upstream-service-time': '76', 'x-pinecone-response-duration-ms': '389', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Metadata size is 61105 bytes, which exceeds the limit of 40960 bytes per vector","details":[]}


In [ ]:
BACKEND = "chroma"  # or "pinecone"

def get_qe():
    return pinecone_qe if BACKEND == "pinecone" else chroma_qe


In [ ]:
def format_citations(response):
    cites = []
    for sn in response.source_nodes:
        md = sn.node.metadata
        reg = md.get("regulation")
        part = md.get("part")
        subpart = md.get("subpart")
        section = md.get("section")
        clause = md.get("clause")
        src = md.get("source_path")

        label_parts = [reg]
        if part:
            label_parts.append(f"Part {part}")
        if subpart:
            label_parts.append(f"Subpart {subpart}")
        if section:
            label_parts.append(f"Section {section}")
        if clause:
            label_parts.append(f"Clause {clause}")

        label = " / ".join([p for p in label_parts if p])
        cites.append(f"{label} ({src})")

    return list(dict.fromkeys(cites))  # dedupe, preserve order


In [ ]:
app = Flask(__name__)

HTML = """
<!doctype html>
<html>
<head>
  <title>Federal Acquisition Retrieval-Augmented Generation (RAG) Chatbot</title>
  <style>
    body { font-family: Arial; margin: 20px; }
    #chat { max-width: 800px; margin: auto; }
    .msg { margin-bottom: 1em; white-space: pre-wrap; }
    .user { font-weight: bold; }
    .bot { margin-left: 1em; }
    textarea { width: 100%; height: 80px; }
    button { margin-top: 10px; padding: 8px 16px; }
  </style>
</head>
<body>
  <div id="chat">
    <h2>FAR / DFARS RAG Chatbot (Qwen + LlamaIndex)</h2>
    <div id="messages"></div>
    <textarea id="question" placeholder="Ask a question..."></textarea>
    <br/>
    <button onclick="send()">Send</button>
  </div>
  <script>
    async function send() {
      const q = document.getElementById('question').value;
      if (!q.trim()) return;
      const m = document.getElementById('messages');
      m.innerHTML += '<div class="msg"><span class="user">You:</span> ' + q + '</div>';
      document.getElementById('question').value = '';

      const res = await fetch('/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ question: q })
      });
      const data = await res.json();
      m.innerHTML += '<div class="msg"><span class="bot">Bot:</span> ' + data.answer + '</div>';
      m.scrollTop = m.scrollHeight;
    }
  </script>
</body>
</html>
"""

@app.route("/")
def index():
    return HTML

@app.route("/chat", methods=["POST"])
def chat():
    q = request.json.get("question", "").strip()
    if not q:
        return jsonify({"answer": "Please enter a question."})

    qe = get_qe()

    system_prompt = (
        "You are a compliance assistant specializing in FAR and DFARS. "
        "Answer ONLY using retrieved context. "
        "If unsure, say you are not certain."
    )

    full_prompt = f"{system_prompt}\n\nUser question: {q}"
    response = qe.query(full_prompt)

    answer_with_cites = format_answer_with_citations(response)
    return jsonify({"answer": answer_with_cites})


In [ ]:
@app.route("/chat_stream", methods=["POST"])
def chat_stream():
    q = request.json.get("question", "").strip()
    if not q:
        return Response("data: Please enter a question.\n\n", mimetype="text/event-stream")

    qe = get_qe()

    system_prompt = (
        "You are a compliance assistant specializing in FAR and DFARS. "
        "Answer ONLY using retrieved context. "
        "If unsure, say you are not certain."
    )
    full_prompt = f"{system_prompt}\n\nUser question: {q}"

    def generate():
        # stream_query yields Response-like chunks
        stream = qe.stream_query(full_prompt)
        for token in stream.response_gen:
            # SSE format: data: <text>\n\n
            yield f"data: {token}\n\n"

        # After completion, send citations as a final JSON payload
        final_response = stream.get_response()
        cites = []
        for i, sn in enumerate(final_response.source_nodes, start=1):
            meta = sn.metadata or {}
            cites.append({
                "index": i,
                "url": meta.get("source_url"),
                "regulation": meta.get("regulation"),
                "part": meta.get("part"),
                "section": meta.get("section"),
            })
        yield f"data: {json.dumps({'citations': cites})}\n\n"

    return Response(generate(), mimetype="text/event-stream")
